## Імпорт бібліотек

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report

## Завантаження даних

In [ ]:
DATA_PATH = "data"

classes = ["idle", "walking", "running", "stairs"]

data = []
labels = []

for label in classes:
    folder = os.path.join(DATA_PATH, label)

    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)

        df = pd.read_csv(file_path)

        df = df[["accelerometer_X",
                 "accelerometer_Y",
                 "accelerometer_Z"]]

        df.columns = ["x", "y", "z"]

        data.append(df)
        labels.append(label)

## Базові ознаки

In [ ]:
def extract_basic_features(df):

    features = []

    for axis in ["x", "y", "z"]:

        values = df[axis]

        features.extend([
            values.mean(),
            values.std(),
            values.min(),
            values.max(),
            values.median()
        ])

    return features

## Розширені ознаки

In [ ]:
def extract_advanced_features(df):

    features = []

    for axis in ["x", "y", "z"]:

        values = df[axis]

        features.extend([
            values.mean(),
            values.std(),
            values.min(),
            values.max(),
            values.median(),
            values.var(),
            values.skew(),
            values.kurtosis(),
            values.max() - values.min()
        ])

    magnitude = np.sqrt(df["x"]**2 +
                        df["y"]**2 +
                        df["z"]**2)

    features.extend([
        magnitude.mean(),
        magnitude.std(),
        magnitude.min(),
        magnitude.max(),
        np.sqrt(np.mean(magnitude**2))
    ])

    return features

## Функція навчання моделей

In [ ]:
def evaluate_models(feature_function, title):

    X = []
    y = []

    for df, label in zip(data, labels):

        X.append(feature_function(df))
        y.append(label)

    X = np.array(X)
    y = np.array(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    print("=" * 60)
    print(title)
    print("=" * 60)

## Алгоритм Random Forest

In [ ]:
    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    rf.fit(X_train, y_train)

    y_pred_rf = rf.predict(X_test)

    print("\nRandom Forest")
    print(classification_report(y_test, y_pred_rf))

## Алгоритм SVM

In [ ]:
    svm = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf"))
    ])

    svm.fit(X_train, y_train)

    y_pred_svm = svm.predict(X_test)

    print("\nSVM")
    print(classification_report(y_test, y_pred_svm))

## Порівняння моделей

In [2]:
evaluate_models(
    extract_basic_features,
    "Базові ознаки"
)

evaluate_models(
    extract_advanced_features,
    "Розширені ознаки"
)

Базові ознаки

Random Forest
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       1.00      0.97      0.98        33
     walking       1.00      1.00      1.00       370

    accuracy                           1.00      1293
   macro avg       1.00      0.99      1.00      1293
weighted avg       1.00      1.00      1.00      1293


SVM
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       0.93      0.76      0.83        33
     walking       0.98      0.99      0.99       370

    accuracy                           0.99      1293
   macro avg       0.98      0.94      0.95      1293
weighted avg       0.99      0.99      0.99      1293

Розширені ознаки

Random Forest
              precision    recall  f1-score   support

        i